# 01 · Camada Bronze — CineData Analytics

**Objetivo:** ingerir os CSVs brutos (Volume) e a cotação do dólar (API do Banco Central) em tabelas **Delta**, no modo **append**, **sem alterar estrutura ou conteúdo**, apenas adicionando a coluna técnica `ingestion_datetime`.

**Decisões de projeto (justificativas):**
- Todas as colunas dos CSVs são lidas como `STRING` (`inferSchema=False`): a Bronze é a cópia fiel da origem; a tipagem é responsabilidade da Silver.
- `multiLine=True` + `escape='"'`: o arquivo possui campos de texto com quebra de linha e aspas duplicadas (padrão RFC-4180).
- `mode=PERMISSIVE`: linhas com *column shift* (mais/menos colunas que o cabeçalho) não derrubam o pipeline; o tratamento é feito na Silver.
- `ingestion_datetime` usa `current_timestamp()` — momento da inserção na Bronze. É a base da deduplicação "versão mais recente" na Silver.
- **API do BCB com mock:** o endpoint `olinda.bcb.gov.br` está instável. O notebook possui o widget `usar_mock`; com `true` (padrão) a resposta é **simulada com o mesmo contrato JSON** da API real (`{"value": [{"cotacaoCompra":..., "dataHoraCotacao":...}]}`). Com `false` o código chama a API real e, se falhar, cai automaticamente no mock (fallback), registrando a origem na coluna `fonte`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import date, datetime, timedelta
import random
import requests

# ---------------------------------------------------------------------------
# Parâmetros (widgets) — permitem reexecutar o notebook sem editar o código
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
dbutils.widgets.text("schema_volume", "default", "Schema do Volume")
dbutils.widgets.text("nome_volume", "inputs", "Nome do Volume")
dbutils.widgets.dropdown("usar_mock", "true", ["true", "false"], "Usar MOCK da API do BCB?")
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-AAAA) — vazio = hoje-7d")
dbutils.widgets.text("data_fim", "", "Data fim (MM-DD-AAAA) — vazio = hoje")

CATALOGO      = dbutils.widgets.get("catalogo")
SCHEMA_VOLUME = dbutils.widgets.get("schema_volume")
NOME_VOLUME   = dbutils.widgets.get("nome_volume")
USAR_MOCK     = dbutils.widgets.get("usar_mock").lower() == "true"

## 1. Ambiente: catálogo, database (schema) Bronze e Volume

In [0]:
spark.sql(f"USE CATALOG `{CATALOGO}`")

# Database (schema) da camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Camada Bronze: dados brutos, sem transformação'")

# Volume que recebe os CSVs da pasta Inputs (upload manual: Catalog > Volume > Upload to this volume)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{SCHEMA_VOLUME}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOGO}`.`{SCHEMA_VOLUME}`.`{NOME_VOLUME}`")

PASTA_VOLUME = f"/Volumes/{CATALOGO}/{SCHEMA_VOLUME}/{NOME_VOLUME}"
print("Volume:", PASTA_VOLUME)
display(dbutils.fs.ls(PASTA_VOLUME))

Volume: /Volumes/workspace/default/inputs


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/inputs/credits_and_tags_IMDB_TMDB.csv,credits_and_tags_IMDB_TMDB.csv,22320678,1791402581000
dbfs:/Volumes/workspace/default/inputs/movies_financials_IMDB_TMDB.csv,movies_financials_IMDB_TMDB.csv,1467743,1791402559000
dbfs:/Volumes/workspace/default/inputs/movies_info_TMDB_IMDB.csv,movies_info_TMDB_IMDB.csv,33924063,1791402583000
dbfs:/Volumes/workspace/default/inputs/movies_metrics_IMDB_TMDB.csv,movies_metrics_IMDB_TMDB.csv,3323884,1791402565000
dbfs:/Volumes/workspace/default/inputs/movies_reviews.csv,movies_reviews.csv,1923431,1791402562000


## 2. Ingestão dos CSVs → tabelas Bronze (Delta, append)

> Observação: os nomes dos arquivos abaixo são os **reais** da pasta Inputs (`movies_info_TMDB_IMDB.csv` e `movies_metrics_IMDB_TMDB.csv`), que diferem levemente do PDF.

In [0]:
MAPA_ARQUIVOS_TABELAS = {
    "movies_info_TMDB_IMDB.csv":        "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv":  "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":     "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":   "bronze.tb_credits_and_tags",
    "movies_reviews.csv":               "bronze.tb_movies_reviews",
}

# Falha cedo e com mensagem clara se algum arquivo não foi enviado ao Volume
arquivos_no_volume = {f.name for f in dbutils.fs.ls(PASTA_VOLUME)}
faltando = [a for a in MAPA_ARQUIVOS_TABELAS if a not in arquivos_no_volume]
if faltando:
    raise FileNotFoundError(f"Arquivos ausentes no Volume {PASTA_VOLUME}: {faltando}")


def ingerir_csv_para_bronze(arquivo: str, tabela: str) -> int:
    """Lê o CSV 'como veio' (tudo STRING), adiciona ingestion_datetime e grava em Delta (append)."""
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", False)      # Bronze = sem tipagem
        .option("multiLine", True)         # textos com quebra de linha (sinopse, comentários)
        .option("quote", '"')
        .option("escape", '"')             # aspas duplicadas ("") dentro de campos
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")      # column shift não interrompe o pipeline
        .load(f"{PASTA_VOLUME}/{arquivo}")
        .withColumn("ingestion_datetime", F.current_timestamp())
    )
    (df.write.format("delta").mode("append").saveAsTable(tabela))
    return spark.table(tabela).count()


for arquivo, tabela in MAPA_ARQUIVOS_TABELAS.items():
    total = ingerir_csv_para_bronze(arquivo, tabela)
    print(f"{arquivo:40s} -> {tabela:32s} | linhas na tabela: {total:,}")

movies_info_TMDB_IMDB.csv                -> bronze.tb_movies_info            | linhas na tabela: 106,596
movies_financials_IMDB_TMDB.csv          -> bronze.tb_movies_financials      | linhas na tabela: 106,165
movies_metrics_IMDB_TMDB.csv             -> bronze.tb_movies_metrics         | linhas na tabela: 103,072
credits_and_tags_IMDB_TMDB.csv           -> bronze.tb_credits_and_tags       | linhas na tabela: 105,608
movies_reviews.csv                       -> bronze.tb_movies_reviews         | linhas na tabela: 32,412


## 3. Ingestão da API do Banco Central (PTAX) → `bronze.tb_cotacao_dolar`

- Formato de data exigido pela API: **MM-DD-AAAA**.
- Janela padrão: últimos 7 dias corridos (a API não retorna fins de semana/feriados).
- **Mock:** `gerar_mock_resposta_bcb()` replica o contrato da API real: vários boletins por dia útil (10h, 11h, 12h e 13h), nenhum boletim em fins de semana/feriados nacionais.

In [0]:
FORMATO_DATA_API = "%m-%d-%Y"

URL_BCB = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    "?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# --- Datas a partir dos widgets (padrão: hoje-7 dias até hoje) -------------
w_ini, w_fim = dbutils.widgets.get("data_inicio").strip(), dbutils.widgets.get("data_fim").strip()
data_fim_dt    = datetime.strptime(w_fim, FORMATO_DATA_API).date() if w_fim else date.today()
data_inicio_dt = datetime.strptime(w_ini, FORMATO_DATA_API).date() if w_ini else data_fim_dt - timedelta(days=7)
if data_inicio_dt > data_fim_dt:
    raise ValueError("data_inicio não pode ser maior que data_fim")

data_inicio_formatada = data_inicio_dt.strftime(FORMATO_DATA_API)
data_fim_formatada    = data_fim_dt.strftime(FORMATO_DATA_API)
print(f"Período: {data_inicio_formatada} -> {data_fim_formatada}")

# Feriados nacionais 2026 sem boletim PTAX (apenas para tornar o mock realista)
FERIADOS_MOCK = {date(2026, 1, 1), date(2026, 4, 3), date(2026, 4, 21), date(2026, 5, 1),
                 date(2026, 6, 4), date(2026, 9, 7), date(2026, 10, 12), date(2026, 11, 2),
                 date(2026, 11, 20), date(2026, 12, 25)}


def gerar_mock_resposta_bcb(data_ini: date, data_fim: date, semente: int = 2026) -> dict:
    """Simula o JSON do endpoint CotacaoDolarPeriodo (mesma estrutura da API real).

    - Só gera boletins em dias úteis (seg-sex, exceto feriados), como a API real.
    - 4 boletins por dia (10:06, 11:06, 12:06, 13:06), cotação em passeio aleatório (semente fixa
      => resultado reprodutível entre execuções).
    """
    rng = random.Random(semente)
    cotacao = 5.40
    registros, dia = [], data_ini
    while dia <= data_fim:
        if dia.weekday() < 5 and dia not in FERIADOS_MOCK:
            for hora in ("10:06", "11:06", "12:06", "13:06"):
                cotacao = round(max(4.5, cotacao + rng.uniform(-0.03, 0.03)), 4)
                registros.append({
                    "cotacaoCompra": cotacao,
                    "dataHoraCotacao": f"{dia:%Y-%m-%d} {hora}:{rng.randint(0, 59):02d}.{rng.randint(0, 999):03d}",
                })
        dia += timedelta(days=1)
    return {
        "@odata.context": "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/$metadata#_CotacaoDolarPeriodo(dataHoraCotacao,cotacaoCompra)",
        "value": registros,
    }


def obter_resposta_bcb(usar_mock: bool):
    """Retorna (json, fonte). Com usar_mock=False tenta a API real e faz fallback para o mock."""
    if not usar_mock:
        url = URL_BCB.format(data_inicio_formatada=data_inicio_formatada, data_fim_formatada=data_fim_formatada)
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            return resp.json(), "API_BCB"
        except Exception as e:  # instabilidade/timeout/403 etc.
            print(f"[AVISO] API do BCB indisponível ({type(e).__name__}: {e}). Usando MOCK como fallback.")
            return gerar_mock_resposta_bcb(data_inicio_dt, data_fim_dt), "MOCK_FALLBACK"
    return gerar_mock_resposta_bcb(data_inicio_dt, data_fim_dt), "MOCK"


resposta, fonte = obter_resposta_bcb(USAR_MOCK)
registros = resposta.get("value", [])
if not registros:
    raise ValueError("Nenhuma cotação retornada para o período (apenas fins de semana/feriados?). Amplie o intervalo.")
print(f"Fonte: {fonte} | boletins recebidos: {len(registros)}")
print("Exemplo de boletim:", registros[0])

Período: 09-30-2026 -> 10-07-2026
Fonte: MOCK | boletins recebidos: 24
Exemplo de boletim: {'cotacaoCompra': 5.3771, 'dataHoraCotacao': '2026-09-30 10:06:32.974'}


In [0]:
# Bronze da cotação: grava o payload como veio (apenas colunas do $select) + metadados de ingestão
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra",   DoubleType(), True),
])

df_cotacao = (
    spark.createDataFrame([(r.get("dataHoraCotacao"), r.get("cotacaoCompra")) for r in registros], schema_cotacao)
    .withColumn("fonte", F.lit(fonte))                         # rastreabilidade: API real x MOCK
    .withColumn("ingestion_datetime", F.current_timestamp())
)

df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")

display(spark.table("bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

dataHoraCotacao,cotacaoCompra,fonte,ingestion_datetime
2026-09-30 10:06:32.974,5.3771,MOCK,2026-10-07T19:54:21.053Z
2026-09-30 11:06:55.975,5.3778,MOCK,2026-10-07T19:54:21.053Z
2026-09-30 12:06:14.916,5.354,MOCK,2026-10-07T19:54:21.053Z
2026-09-30 13:06:35.430,5.3601,MOCK,2026-10-07T19:54:21.053Z
2026-10-01 10:06:35.862,5.3771,MOCK,2026-10-07T19:54:21.053Z
2026-10-01 11:06:49.502,5.3909,MOCK,2026-10-07T19:54:21.053Z
2026-10-01 12:06:37.451,5.406,MOCK,2026-10-07T19:54:21.053Z
2026-10-01 13:06:39.082,5.3904,MOCK,2026-10-07T19:54:21.053Z
2026-10-02 10:06:52.100,5.367,MOCK,2026-10-07T19:54:21.053Z
2026-10-02 11:06:52.912,5.364,MOCK,2026-10-07T19:54:21.053Z


## 4. Validação final da Bronze

In [0]:
for tabela in list(MAPA_ARQUIVOS_TABELAS.values()) + ["bronze.tb_cotacao_dolar"]:
    df = spark.table(tabela)
    print(f"{tabela:32s} linhas={df.count():>9,}  colunas={len(df.columns)}  (última coluna: {df.columns[-1]})")

bronze.tb_movies_info            linhas=  106,596  colunas=11  (última coluna: ingestion_datetime)
bronze.tb_movies_financials      linhas=  106,165  colunas=4  (última coluna: ingestion_datetime)
bronze.tb_movies_metrics         linhas=  103,072  colunas=7  (última coluna: ingestion_datetime)
bronze.tb_credits_and_tags       linhas=  105,608  colunas=10  (última coluna: ingestion_datetime)
bronze.tb_movies_reviews         linhas=   32,412  colunas=5  (última coluna: ingestion_datetime)
bronze.tb_cotacao_dolar          linhas=       24  colunas=4  (última coluna: ingestion_datetime)
